# Azalyst v4.0 — Notebook 2: Train + Walk-Forward

**Pre-requisite:** Run `azalyst_1_feature_cache.ipynb` first and confirm all ~444 `.parquet` files are in your Google Drive folder.

**What this does (v4 Architecture):**
- Downloads the full feature cache from **Google Drive** into `/kaggle/working/feature_cache/`
- Trains XGBoost base model + Meta-labeling model on Year 1 (expanding window start)
- Runs walk-forward simulation on **Year 2 + Year 3** (2-year OOS)
- Expanding window retraining every 13 weeks (Y1 → Y1+Y2 → Y1+Y2+Y3)
- **Regime-aware IC feature selection** — drops features with negative rolling IC
- **Risk integration** — VaR/CVaR position sizing
- **Drawdown kill-switch** — halts at -15% max DD
- **SHAP explainability** after every training cycle
- Saves results, charts, and ZIP to `/kaggle/working/azalyst_output/`

---
### Setup
1. In Kaggle Notebook → **Add-ons → Secrets** → add secret named `GDRIVE_SERVICE_KEY`  
   *(same JSON key used in Notebook 1 — reuse it)*
2. Set `GDRIVE_FOLDER_ID` in Cell 1 below to your Drive folder ID
3. Enable **GPU T4** in notebook settings

---
Storage:
- Download (feature cache from Drive): ~9GB on disk  
- Output (models + results): ~500MB — well under 20GB limit

In [ ]:
# ── Cell 0: Install deps ─────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'sortedcontainers', 'shap',
            'google-api-python-client', 'google-auth']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Deps installed.')

In [ ]:
# ── Cell 1: Imports + Paths (v4) ──────────────────────────────────────────────
import os, sys, time, json, gc, pickle, warnings, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sp_stats
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score
import psutil
warnings.filterwarnings('ignore')

# ── SET THIS ──────────────────────────────────────────────────────────────────
GDRIVE_FOLDER_ID  = 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE'  # same as Notebook 1
# ─────────────────────────────────────────────────────────────────────────────

FEATURE_CACHE_DIR = '/kaggle/working/feature_cache'
RESULTS_DIR       = '/kaggle/working/azalyst_output'

for d in [FEATURE_CACHE_DIR, RESULTS_DIR, f'{RESULTS_DIR}/models', f'{RESULTS_DIR}/shap']:
    os.makedirs(d, exist_ok=True)

print(f'Feature cache  : {FEATURE_CACHE_DIR}')
print(f'Results dir    : {RESULTS_DIR}')
print(f'RAM available  : {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'Disk free      : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Download Feature Cache from Google Drive ─────────────────────────
import json as _json
from kaggle_secrets import UserSecretsClient
from google.oauth2 import service_account
from googleapiclient.discovery import build
import io
from googleapiclient.http import MediaIoBaseDownload

_secret = UserSecretsClient().get_secret('GDRIVE_SERVICE_KEY')
_creds  = service_account.Credentials.from_service_account_info(
    _json.loads(_secret),
    scopes=['https://www.googleapis.com/auth/drive.readonly']
)
drive = build('drive', 'v3', credentials=_creds)
print('✅ [Google Drive] Authenticated')

# List all parquet files in the Drive folder
def list_drive_files():
    results, page_token = [], None
    while True:
        resp = drive.files().list(
            q=f"'{GDRIVE_FOLDER_ID}' in parents and trashed=false and name contains '.parquet'",
            fields='nextPageToken, files(id, name)',
            pageToken=page_token
        ).execute()
        results.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return results

drive_files = list_drive_files()
print(f'  Files in Drive folder : {len(drive_files)}')

# Download each file (skip if already on disk)
t0 = time.time()
downloaded = skipped = 0
for i, f in enumerate(drive_files, 1):
    dest = Path(FEATURE_CACHE_DIR) / f['name']
    if dest.exists():
        skipped += 1
        continue
    request = drive.files().get_media(fileId=f['id'])
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    dest.write_bytes(buf.getvalue())
    downloaded += 1
    if i % 50 == 0 or i == len(drive_files):
        disk_used = shutil.disk_usage('/kaggle/working').used / 1e9
        print(f'  [{i}/{len(drive_files)}] downloaded={downloaded} skipped={skipped} '
              f'disk={disk_used:.1f}GB elapsed={( time.time()-t0)/60:.1f}min')

cache_files = sorted(Path(FEATURE_CACHE_DIR).glob('*.parquet'))
print(f'\n✅ Download complete — {len(cache_files)} files ready in {FEATURE_CACHE_DIR}')
if not cache_files:
    raise RuntimeError('No parquet files downloaded. Check GDRIVE_FOLDER_ID and service account permissions.')


In [ ]:
# ── Cell 3: GPU Setup ─────────────────────────────────────────────────────────
import subprocess, xgboost as xgb

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['CUDA_DEVICE_ORDER']    = 'PCI_BUS_ID'

try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=5)
    print(r.stdout.decode()[:400])
except: pass

def detect_cuda_api():
    try:
        _x = np.random.rand(1000, 10).astype(np.float32)
        _y = np.random.randint(0, 2, 1000)
        try:
            xgb.XGBClassifier(device='cuda', n_estimators=5, verbosity=0).fit(_x, _y)
            return 'new'
        except: pass
        try:
            xgb.XGBClassifier(tree_method='gpu_hist', n_estimators=5, verbosity=0).fit(_x, _y)
            return 'old'
        except: return None
    except: return None

CUDA_API = detect_cuda_api()
HAS_GPU  = CUDA_API is not None
print(f'XGBoost {xgb.__version__}: CUDA_API={CUDA_API} | HAS_GPU={HAS_GPU}')

def mem_gb(): return psutil.virtual_memory().used / 1e9
def aggressive_gc(): gc.collect()

In [ ]:
# ── Cell 4: v4 Config ─────────────────────────────────────────────────────────
FEATURE_COLS = [
    'ret_1bar','ret_1h','ret_4h','ret_1d','ret_2d','ret_3d','ret_1w',
    'vol_ratio','vol_ret_1h','vol_ret_1d','obv_change','vpt_change','vol_momentum',
    'rvol_1h','rvol_4h','rvol_1d','vol_ratio_1h_1d','atr_norm','parkinson_vol','garman_klass',
    'rsi_14','rsi_6','macd_hist','bb_pos','bb_width','stoch_k','stoch_d','cci_14','adx_14','dmi_diff',
    'vwap_dev','amihud','kyle_lambda','spread_proxy','body_ratio','candle_dir',
    'wick_top','wick_bot','price_accel','skew_1d','kurt_1d','max_ret_4h',
    'wq_alpha001','wq_alpha012','wq_alpha031','wq_alpha098',
    'cs_momentum','cs_reversal','vol_adjusted_mom','trend_consistency',
    'vol_regime','trend_strength','corr_btc_proxy','hurst_exp','fft_strength',
    'frac_diff_close',
]

RETRAIN_WEEKS   = 13       # quarterly
TOP_QUANTILE    = 0.15     # top/bottom 15% for long/short
FEE_RATE        = 0.001    # 0.1% per leg
ROUND_TRIP_FEE  = FEE_RATE * 2
HORIZON_BARS    = 48       # 4H @ 5-min
MAX_TRAIN_ROWS  = 4_000_000
MAX_LOAD_ROWS   = 6_000_000
KAGGLE_RAM_GB   = 30

# v4 new config
MAX_DRAWDOWN_KILL   = -0.15       # kill-switch: halt if DD > 15%
IC_SELECTION_THRESH = -0.02       # drop features with rolling IC below this
IC_LOOKBACK_WEEKS   = 8           # rolling window for feature IC estimation
MIN_FEATURES        = 20          # never drop below this many features
VAR_CONFIDENCE      = 0.95
POSITION_RISK_CAP   = 0.03        # max 3% portfolio risk per position

print(f'[Cell 4] v4 Config loaded.')
print(f'  Features       : {len(FEATURE_COLS)}')
print(f'  Retrain every  : {RETRAIN_WEEKS} weeks')
print(f'  Fee round-trip : {ROUND_TRIP_FEE*100:.2f}%')
print(f'  Max DD kill    : {MAX_DRAWDOWN_KILL*100:.0f}%')
print(f'  IC thresh      : {IC_SELECTION_THRESH}')
print(f'  VRAM guard     : {MAX_TRAIN_ROWS:,} rows')
print(f'  Train strategy : Expanding window (Y1 -> Y1+Y2 -> Y1+Y2+Y3)')
print(f'  Walk strategy  : Y2 + Y3 (2-year OOS)')

In [ ]:
# ── Cell 5: Load Feature Store ────────────────────────────────────────────────
print(f'[Cell 5] Loading {len(cache_files)} symbol files from feature cache...')
t0 = time.time()

# Memory guard: stride if total rows would exceed MAX_LOAD_ROWS
_probe      = pd.read_parquet(cache_files[0])
_rows_each  = len(_probe)
_total_est  = _rows_each * len(cache_files)
LOAD_STRIDE = max(1, _total_est // MAX_LOAD_ROWS)
if LOAD_STRIDE > 1:
    print(f'  [MEM GUARD] ~{_total_est:,} est. rows -> LOAD_STRIDE={LOAD_STRIDE}')
del _probe; gc.collect()

frames = []
for fp in cache_files:
    try:
        df = pd.read_parquet(fp)
        if not isinstance(df.index, pd.DatetimeIndex):
            df.index = pd.to_datetime(df.index, unit='ms', utc=True)
        elif df.index.tz is None:
            df.index = df.index.tz_localize('UTC')
        if 'symbol' not in df.columns:
            df['symbol'] = fp.stem
        if LOAD_STRIDE > 1:
            df = df.iloc[::LOAD_STRIDE]
        frames.append(df)
    except Exception as e:
        print(f'  [WARN] {fp.stem}: {e}')

if not frames:
    raise RuntimeError('No feature files loaded.')

ALL_DATA = pd.concat(frames, axis=0).sort_index()
del frames; gc.collect()

print(f'  Pooled    : {len(ALL_DATA):,} rows | {ALL_DATA["symbol"].nunique()} symbols')
print(f'  Date range: {ALL_DATA.index.min().date()} -> {ALL_DATA.index.max().date()}')
print(f'  RAM used  : {mem_gb():.1f} GB / {KAGGLE_RAM_GB} GB')
print(f'  Loaded in : {time.time()-t0:.1f}s')

# Validate all feature columns present
missing = [c for c in FEATURE_COLS if c not in ALL_DATA.columns]
if missing:
    raise ValueError(f'Cache missing columns: {missing}. Re-run Notebook 1.')
print(f'  All {len(FEATURE_COLS)} feature columns present')

In [ ]:
# ── Cell 6: v4 Date Splits (Y1 / Y2 / Y3) ────────────────────────────────────
print('[Cell 6] v4 Date splits — 3-way Y1/Y2/Y3...')

global_min  = ALL_DATA.index.min()
global_max  = ALL_DATA.index.max()
total_span  = global_max - global_min
Y1_END      = global_min + (total_span / 3)
Y2_END      = global_min + (total_span * 2 / 3)

print(f'  Year 1 (init train): {global_min.date()} -> {Y1_END.date()}')
print(f'  Year 2 (walk)      : {Y1_END.date()} -> {Y2_END.date()}')
print(f'  Year 3 (walk)      : {Y2_END.date()} -> {global_max.date()}')
print(f'  v4 walk range      : Y2+Y3 = 2-year OOS')

# Build initial training matrix (Y1 only — expanding window start)
print(f'\n  Building Y1 training matrix...')
train_df = ALL_DATA[ALL_DATA.index < Y1_END].copy()

if 'future_ret' in train_df.columns:
    train_df['alpha_label'] = (
        train_df.groupby(train_df.index)['future_ret']
        .transform(lambda x: (x > x.median()).astype(float))
    )
else:
    raise KeyError('future_ret missing from cache. Re-run Notebook 1.')

for c in FEATURE_COLS:
    if c not in train_df.columns: train_df[c] = 0.0

feat_matrix = train_df[FEATURE_COLS].values.astype(np.float32)
label_arr   = train_df['alpha_label'].values.astype(np.float32)
ret_arr     = train_df['future_ret'].values.astype(np.float32)

valid_mask  = np.isfinite(feat_matrix).all(axis=1) & np.isfinite(label_arr)
feat_matrix, label_arr, ret_arr = feat_matrix[valid_mask], label_arr[valid_mask], ret_arr[valid_mask]

if len(feat_matrix) > MAX_TRAIN_ROWS:
    idx = np.sort(np.random.choice(len(feat_matrix), MAX_TRAIN_ROWS, replace=False))
    feat_matrix, label_arr, ret_arr = feat_matrix[idx], label_arr[idx], ret_arr[idx]

del train_df; gc.collect()

X_TRAIN, Y_TRAIN, Y_RET = feat_matrix, label_arr, ret_arr
print(f'  Training matrix : {X_TRAIN.shape[0]:,} rows x {X_TRAIN.shape[1]} features')
print(f'  Label balance   : {Y_TRAIN.mean()*100:.1f}% positive (target ~50%)')

In [ ]:
# ── Cell 7: v4 Model Definitions (inlined for Kaggle) ─────────────────────────
import xgboost as xgb
from scipy import stats as sp_stats
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import roc_auc_score

def compute_ic(y_pred, y_true):
    mask = np.isfinite(y_pred) & np.isfinite(y_true)
    if mask.sum() < 10: return 0.0
    return float(sp_stats.spearmanr(y_pred[mask], y_true[mask])[0])

def make_xgb_params(n_est=1000, max_depth=6, min_child_weight=30):
    p = dict(
        n_estimators=n_est, learning_rate=0.02, max_depth=max_depth,
        min_child_weight=min_child_weight, subsample=0.8,
        colsample_bytree=0.7, colsample_bylevel=0.7,
        reg_alpha=0.1, reg_lambda=1.0, eval_metric='auc',
        early_stopping_rounds=50, verbosity=0, random_state=42,
    )
    if CUDA_API == 'new':  p['device'] = 'cuda'
    elif CUDA_API == 'old': p['tree_method'] = 'gpu_hist'
    return p

class PurgedTimeSeriesCV:
    def __init__(self, n_splits=5, gap=48):
        self.n_splits = n_splits; self.gap = gap
    def split(self, X):
        n = len(X); fold_size = n // (self.n_splits + 1)
        for i in range(self.n_splits):
            train_end = (i + 1) * fold_size
            val_start = train_end + self.gap
            val_end   = val_start + fold_size
            if val_end > n: break
            yield np.arange(0, train_end), np.arange(val_start, val_end)

def train_model(X, y, y_ret=None, features_used=None, label=''):
    features_used = features_used or FEATURE_COLS
    scaler = RobustScaler()
    Xs = scaler.fit_transform(X)
    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    aucs, ics = [], []
    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2: continue
        m = xgb.XGBClassifier(**make_xgb_params())
        m.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)
        probs = m.predict_proba(Xs[val])[:, 1]
        try: aucs.append(roc_auc_score(y[val], probs))
        except: pass
        if y_ret is not None:
            mask = np.isfinite(probs) & np.isfinite(y_ret[val])
            if mask.sum() >= 10:
                ics.append(float(sp_stats.spearmanr(probs[mask], y_ret[val][mask])[0]))
    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    mean_ic  = float(np.mean(ics))  if ics  else 0.0
    icir     = float(np.mean(ics) / (np.std(ics)+1e-8)) if len(ics) > 1 else 0.0
    final = xgb.XGBClassifier(**make_xgb_params())
    split = int(len(Xs) * 0.9)
    final.fit(Xs[:split], y[:split], eval_set=[(Xs[split:], y[split:])], verbose=False)
    importance = pd.Series(final.feature_importances_, index=features_used,
                           name='importance').sort_values(ascending=False)
    return final, scaler, importance, mean_auc, mean_ic, icir

def train_meta_model(primary_model, primary_scaler, X, y, features_used=None, label='meta'):
    features_used = features_used or FEATURE_COLS
    Xs = primary_scaler.transform(X)
    cv = PurgedTimeSeriesCV(n_splits=5, gap=48)
    oos_preds = np.full(len(y), np.nan, dtype=np.float32)
    for fold, (tr, val) in enumerate(cv.split(Xs), 1):
        if len(np.unique(y[val])) < 2: continue
        m_temp = xgb.XGBClassifier(**make_xgb_params())
        m_temp.fit(Xs[tr], y[tr], eval_set=[(Xs[val], y[val])], verbose=False)
        oos_preds[val] = m_temp.predict_proba(Xs[val])[:, 1]
    valid = np.isfinite(oos_preds)
    if valid.sum() < 200:
        print(f'  [{label}] Not enough OOS data ({valid.sum()}) — skipping meta')
        return None, None
    meta_y   = ((oos_preds[valid] >= 0.5).astype(float) == y[valid]).astype(float)
    X_meta   = np.column_stack([Xs[valid], oos_preds[valid]])
    meta_scaler = RobustScaler()
    X_meta_s = meta_scaler.fit_transform(X_meta)
    meta_params = make_xgb_params(n_est=500, max_depth=4, min_child_weight=50)
    meta = xgb.XGBClassifier(**meta_params)
    split = int(len(X_meta_s) * 0.9)
    meta.fit(X_meta_s[:split], meta_y[:split],
             eval_set=[(X_meta_s[split:], meta_y[split:])], verbose=False)
    val_acc = float((meta.predict(X_meta_s[split:]) == meta_y[split:]).mean())
    print(f'  [{label}] Meta accuracy: {val_acc*100:.1f}% on {valid.sum():,} OOS rows')
    return meta, meta_scaler

# ── v4 NEW FUNCTIONS ──

def detect_regime(all_data, week_end):
    """4-state regime detector: BULL_TREND, BEAR_TREND, HIGH_VOL_LATERAL, LOW_VOL_GRIND."""
    btc_mask = all_data['symbol'] == 'BTCUSDT' if 'symbol' in all_data.columns else pd.Series(False, index=all_data.index)
    btc = all_data[btc_mask]
    if len(btc) == 0:
        return 'LOW_VOL_GRIND'
    lookback = btc[btc.index < week_end].tail(4 * 288)
    if len(lookback) < 288:
        return 'LOW_VOL_GRIND'
    rvol = lookback.get('rvol_1d')
    avg_vol = float(rvol.dropna().mean()) if rvol is not None and len(rvol.dropna()) > 0 else 0.02
    recent_vol = float(rvol.dropna().tail(288).mean()) if rvol is not None and len(rvol.dropna()) > 288 else avg_vol
    ret_1w = lookback.get('ret_1w')
    btc_ret = float(ret_1w.dropna().iloc[-1]) if ret_1w is not None and len(ret_1w.dropna()) > 0 else 0.0
    high_vol = recent_vol > avg_vol * 1.3
    if btc_ret > 0.03 and not high_vol: return 'BULL_TREND'
    elif btc_ret < -0.03:               return 'BEAR_TREND'
    elif high_vol:                       return 'HIGH_VOL_LATERAL'
    else:                                return 'LOW_VOL_GRIND'

def compute_feature_ic(all_data, week_start, week_end, features):
    """Cross-sectional Spearman IC per feature for a single week."""
    week_data = all_data[(all_data.index >= week_start) & (all_data.index < week_end)]
    if 'symbol' not in week_data.columns or 'future_ret' not in week_data.columns:
        return {f: 0.0 for f in features}
    syms = week_data['symbol'].unique()
    if len(syms) < 20:
        return {f: 0.0 for f in features}
    sym_stats = {}
    for sym in syms:
        s = week_data[week_data['symbol'] == sym]
        if len(s) < 3: continue
        row = {}
        for feat in features:
            if feat in s.columns:
                row[feat] = float(s[feat].dropna().mean()) if len(s[feat].dropna()) > 0 else np.nan
        row['_ret'] = float(s['future_ret'].dropna().mean()) if len(s['future_ret'].dropna()) > 0 else np.nan
        sym_stats[sym] = row
    if len(sym_stats) < 20:
        return {f: 0.0 for f in features}
    df_ic = pd.DataFrame(sym_stats).T
    result = {}
    for feat in features:
        if feat in df_ic.columns:
            valid = df_ic[[feat, '_ret']].dropna()
            if len(valid) >= 20:
                ic, _ = sp_stats.spearmanr(valid[feat], valid['_ret'])
                result[feat] = float(ic) if np.isfinite(ic) else 0.0
            else:
                result[feat] = 0.0
        else:
            result[feat] = 0.0
    return result

def select_features_by_ic(ic_history, all_features):
    """Regime-aware feature selection: drop features with consistently negative IC."""
    if not ic_history:
        return list(all_features)
    scores = {}
    for feat in all_features:
        ics = ic_history.get(feat, [])
        if len(ics) >= 4:
            scores[feat] = float(np.mean(ics[-IC_LOOKBACK_WEEKS:]))
        else:
            scores[feat] = 0.0
    selected = [f for f, ic in scores.items() if ic >= IC_SELECTION_THRESH]
    if len(selected) < MIN_FEATURES:
        ranked = sorted(scores.items(), key=lambda x: -x[1])
        selected = [f for f, _ in ranked[:MIN_FEATURES]]
    return selected

def compute_shap(model, X_sample, feature_names, max_samples=5000):
    """Compute SHAP values using TreeExplainer."""
    try:
        import shap
    except ImportError:
        print('  [SHAP] shap not installed — skipping')
        return {}
    if len(X_sample) > max_samples:
        idx = np.random.choice(len(X_sample), max_samples, replace=False)
        X_sample = X_sample[idx]
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        mean_abs = np.abs(shap_values).mean(axis=0)
        result = {feat: float(val) for feat, val in zip(feature_names, mean_abs)}
        top_feat = max(result, key=result.get)
        print(f'  [SHAP] {len(feature_names)} features (top: {top_feat}={result[top_feat]:.4f})')
        return result
    except Exception as e:
        print(f'  [SHAP] Error: {e}')
        return {}

def compute_drawdown(weekly_returns):
    """Returns current max drawdown (negative number)."""
    if not weekly_returns: return 0.0
    cum = np.cumprod([1 + r for r in weekly_returns])
    peak = np.maximum.accumulate(cum)
    dd = (cum - peak) / peak
    return float(dd.min())

def predict_week(model, scaler, all_data, ws, we, features_used, meta_model=None, meta_scaler=None):
    """Predict cross-sectional probabilities for a single week."""
    week_df = all_data[(all_data.index >= ws) & (all_data.index < we)].copy()
    if len(week_df) < 10:
        return {}, {}, {}
    predictions, actual_rets, meta_confs = {}, {}, {}
    if 'symbol' not in week_df.columns:
        return {}, {}, {}
    for sym in week_df['symbol'].unique():
        s = week_df[week_df['symbol'] == sym]
        if len(s) < 3: continue
        for c in features_used:
            if c not in s.columns: s[c] = 0.0
        feat = s[features_used].values.astype(np.float32)
        valid = np.isfinite(feat).all(axis=1)
        if valid.sum() < 2: continue
        try:
            feat_scaled = scaler.transform(feat[valid])
            probs = model.predict_proba(feat_scaled)[:, 1]
            predictions[sym] = float(probs.mean())
            if meta_model is not None and meta_scaler is not None:
                try:
                    meta_input = np.column_stack([feat_scaled, probs.reshape(-1, 1)])
                    meta_scaled = meta_scaler.transform(meta_input)
                    meta_probs = meta_model.predict_proba(meta_scaled)[:, 1]
                    meta_confs[sym] = float(meta_probs.mean())
                except: pass
            if 'future_ret' in s.columns:
                ret_col = s['future_ret'].values[valid]
                finite = ret_col[np.isfinite(ret_col)]
                if len(finite) > 0:
                    actual_rets[sym] = float(finite.mean())
        except: pass
    return predictions, actual_rets, meta_confs

def simulate_weekly_trades(predictions, actual_rets, prev_longs, prev_shorts,
                           meta_confs=None, weekly_returns_hist=None):
    """Position-tracked fees + meta-labeling confidence + risk-adjusted sizing."""
    if not predictions:
        return [], 0.0, set(), set()
    pred_series = pd.Series(predictions)
    ranked = pred_series.rank(pct=True)
    cur_longs  = set(ranked[ranked >= (1 - TOP_QUANTILE)].index)
    cur_shorts = set(ranked[ranked <= TOP_QUANTILE].index)
    # VaR-based risk scaling
    risk_scale = 1.0
    if weekly_returns_hist and len(weekly_returns_hist) >= 4:
        ret_s = pd.Series(weekly_returns_hist)
        var_val = float(ret_s.quantile(1 - VAR_CONFIDENCE))
        if var_val < -POSITION_RISK_CAP:
            risk_scale = max(0.3, POSITION_RISK_CAP / abs(var_val))
    trades = []
    for sym in cur_longs:
        ret = actual_rets.get(sym, 0.0)
        if not np.isfinite(ret): ret = 0.0
        fee = 0.0 if sym in prev_longs else ROUND_TRIP_FEE
        meta = meta_confs.get(sym, 1.0) if meta_confs else 1.0
        sized = meta * risk_scale
        trades.append({'symbol': sym, 'signal': 'BUY', 'pred_prob': round(predictions[sym], 5),
                       'pnl_percent': round((ret - fee) * sized * 100, 4),
                       'raw_ret_pct': round(ret * 100, 4), 'meta_size': round(sized, 4)})
    for sym in cur_shorts:
        ret = actual_rets.get(sym, 0.0)
        if not np.isfinite(ret): ret = 0.0
        fee = 0.0 if sym in prev_shorts else ROUND_TRIP_FEE
        meta = meta_confs.get(sym, 1.0) if meta_confs else 1.0
        sized = meta * risk_scale
        trades.append({'symbol': sym, 'signal': 'SELL', 'pred_prob': round(predictions[sym], 5),
                       'pnl_percent': round((-ret - fee) * sized * 100, 4),
                       'raw_ret_pct': round(ret * 100, 4), 'meta_size': round(sized, 4)})
    if trades:
        sizes = np.array([t['meta_size'] for t in trades])
        pnls  = np.array([t['pnl_percent'] for t in trades])
        week_ret = float(np.average(pnls, weights=sizes)) / 100
    else:
        week_ret = 0.0
    return trades, week_ret, cur_longs, cur_shorts

print('[Cell 7] v4 Model definitions ready.')
print(f'  GPU: {HAS_GPU} (CUDA_API={CUDA_API})')
print(f'  Functions: train_model, train_meta_model, detect_regime, compute_feature_ic,')
print(f'             select_features_by_ic, compute_shap, compute_drawdown,')
print(f'             predict_week, simulate_weekly_trades')

In [ ]:
# ── Cell 8: v4 Initial Training on Y1 (Expanding Window Start) ────────────────
print('[Cell 8] Training initial model on Y1 data...')
print(f'  Train window: {global_min.date()} -> {Y1_END.date()}')
print(f'  GPU: {HAS_GPU} (CUDA_API={CUDA_API})')

active_features = list(FEATURE_COLS)

# Train base model on Y1
t0 = time.time()
BASE_MODEL, BASE_SCALER, importance, mean_auc, mean_ic, icir = train_model(
    X_TRAIN, Y_TRAIN, Y_RET, features_used=active_features, label='base_y1'
)
elapsed = time.time() - t0

# Save base model
BASE_MODEL.save_model(f'{RESULTS_DIR}/models/model_v4_base.json')
with open(f'{RESULTS_DIR}/models/scaler_v4_base.pkl', 'wb') as f:
    pickle.dump(BASE_SCALER, f)
importance.to_csv(f'{RESULTS_DIR}/feature_importance_v4_base.csv')

print(f'\n  -- Base Model (Y1) --')
print(f'  AUC (CV)  : {mean_auc:.4f}')
print(f'  IC  (CV)  : {mean_ic:.4f}')
print(f'  ICIR      : {icir:.4f}')
print(f'  Elapsed   : {elapsed:.1f}s')
print(f'  Top 5: {list(importance.head(5).index)}')

# SHAP
print('\n  Computing SHAP values...')
Xs_shap = BASE_SCALER.transform(X_TRAIN)
shap_vals = compute_shap(BASE_MODEL, Xs_shap, active_features)
if shap_vals:
    shap_dir = f'{RESULTS_DIR}/shap'
    os.makedirs(shap_dir, exist_ok=True)
    shap_df = pd.DataFrame([
        {'feature': k, 'mean_abs_shap': v, 'rank': i+1}
        for i, (k, v) in enumerate(sorted(shap_vals.items(), key=lambda x: -x[1]))
    ])
    shap_df.to_csv(f'{shap_dir}/shap_importance_v4_base.csv', index=False)
    print(f'  SHAP saved -> {shap_dir}/shap_importance_v4_base.csv')

# Meta model
print('\n  Training meta-labeling model...')
META_MODEL, META_SCALER = train_meta_model(
    BASE_MODEL, BASE_SCALER, X_TRAIN, Y_TRAIN,
    features_used=active_features, label='meta_y1'
)
if META_MODEL is not None:
    with open(f'{RESULTS_DIR}/models/meta_v4_base.pkl', 'wb') as f:
        pickle.dump(META_MODEL, f)
    with open(f'{RESULTS_DIR}/models/meta_scaler_v4_base.pkl', 'wb') as f:
        pickle.dump(META_SCALER, f)

with open(f'{RESULTS_DIR}/train_summary_v4.json', 'w') as f:
    json.dump({'mean_auc': round(mean_auc,5), 'mean_ic': round(mean_ic,5),
               'icir': round(icir,5), 'n_rows': int(len(X_TRAIN)),
               'n_features': len(active_features), 'use_gpu': HAS_GPU}, f, indent=2)

del Xs_shap
gc.collect()
print(f'\n[Cell 8] Done. RAM: {mem_gb():.1f} GB')

In [ ]:
# ── Cell 9: v4 Walk-Forward Y2+Y3 (Expanding Window + IC Selection + Kill-Switch) ─
t0_wf = time.time()
print('[Cell 9] Starting v4 walk-forward...')
print(f'  Walk: Y2 ({Y1_END.date()}) + Y3 ({Y2_END.date()}) -> {global_max.date()}')
print(f'  Retrain: every {RETRAIN_WEEKS} weeks (expanding window)')
print(f'  Kill-switch: {MAX_DRAWDOWN_KILL*100:.0f}% max DD')
print(f'  Meta: {"enabled" if META_MODEL is not None else "disabled"}')
print()

weeks = pd.date_range(start=Y1_END, end=global_max, freq='W-MON')
if len(weeks) < 3:
    raise RuntimeError('Not enough walk-forward weeks.')

current_model  = BASE_MODEL
current_scaler = BASE_SCALER
current_meta   = META_MODEL
current_meta_scaler = META_SCALER
retrain_count  = 0

all_trades_list      = []
weekly_summary_list  = []
weekly_returns_hist  = []
prev_longs, prev_shorts = set(), set()
feature_ic_history = {}
kill_switch_active = False
kill_switch_hit    = False

for week_num, (ws, we) in enumerate(zip(weeks[:-1], weeks[1:]), 1):
    # Kill-switch check
    current_dd = compute_drawdown(weekly_returns_hist)
    if current_dd < MAX_DRAWDOWN_KILL:
        print(f'\n  *** KILL SWITCH *** Week {week_num}: DD={current_dd*100:.1f}%')
        print('  Halting all trading. Pausing for 4 weeks...')
        kill_switch_hit = True
        for skip in range(min(4, len(weeks) - week_num - 1)):
            weekly_returns_hist.append(0.0)
            weekly_summary_list.append({
                'week': week_num + skip, 'week_start': str(ws.date()),
                'week_end': str(we.date()), 'n_symbols': 0, 'n_trades': 0,
                'week_return_pct': 0.0, 'ic': 0.0, 'turnover_pct': 0.0,
                'regime': 'KILL_SWITCH', 'retrained': False,
            })
        continue

    # Regime detection
    regime = detect_regime(ALL_DATA, we)

    # Feature IC tracking (every 2 weeks)
    if week_num > 1 and week_num % 2 == 0:
        fic = compute_feature_ic(ALL_DATA, ws, we, active_features)
        for feat, ic_val in fic.items():
            if feat not in feature_ic_history:
                feature_ic_history[feat] = []
            feature_ic_history[feat].append(ic_val)

    # Quarterly retrain with expanding window
    did_retrain = False
    if week_num % RETRAIN_WEEKS == 0:
        print(f'  Week {week_num:3d}: EXPANDING RETRAIN (data up to {we.date()})...')
        try:
            # Regime-aware feature selection
            new_features = select_features_by_ic(feature_ic_history, FEATURE_COLS)
            n_dropped = len(active_features) - len(new_features)
            if n_dropped != 0:
                print(f'    IC filter: {len(active_features)} -> {len(new_features)} features')
            active_features = new_features

            # Expanding window: build training matrix from all data up to now
            rt_df = ALL_DATA[ALL_DATA.index < we].copy()
            for c in active_features:
                if c not in rt_df.columns: rt_df[c] = 0.0
            if 'future_ret' in rt_df.columns:
                rt_df['alpha_label'] = rt_df.groupby(rt_df.index)['future_ret'].transform(
                    lambda x: (x > x.median()).astype(float))
            Xr = rt_df[active_features].values.astype(np.float32)
            yr = rt_df['alpha_label'].values.astype(np.float32)
            yr_r = rt_df['future_ret'].values.astype(np.float32)
            vm = np.isfinite(Xr).all(axis=1) & np.isfinite(yr)
            Xr, yr, yr_r = Xr[vm], yr[vm], yr_r[vm]
            if len(Xr) > MAX_TRAIN_ROWS:
                idx2 = np.sort(np.random.choice(len(Xr), MAX_TRAIN_ROWS, replace=False))
                Xr, yr, yr_r = Xr[idx2], yr[idx2], yr_r[idx2]

            m_new, s_new, imp_new, auc_n, ic_n, icir_n = train_model(
                Xr, yr, yr_r, features_used=active_features, label=f'v4_w{week_num:03d}'
            )
            current_model, current_scaler = m_new, s_new
            retrain_count += 1
            m_new.save_model(f'{RESULTS_DIR}/models/model_v4_week{week_num:03d}.json')
            imp_new.to_csv(f'{RESULTS_DIR}/feature_importance_v4_week{week_num:03d}.csv')
            print(f'    AUC={auc_n:.4f}  IC={ic_n:.4f}  ICIR={icir_n:.4f}')

            # SHAP on retrained model
            Xs_rt = s_new.transform(Xr)
            shap_v = compute_shap(m_new, Xs_rt, active_features)
            if shap_v:
                shap_dir = f'{RESULTS_DIR}/shap'
                os.makedirs(shap_dir, exist_ok=True)
                shap_df = pd.DataFrame([
                    {'feature': k, 'mean_abs_shap': v, 'rank': i+1}
                    for i, (k, v) in enumerate(sorted(shap_v.items(), key=lambda x: -x[1]))
                ])
                shap_df.to_csv(f'{shap_dir}/shap_importance_v4_week{week_num:03d}.csv', index=False)

            # Retrain meta
            meta_new, meta_s_new = train_meta_model(
                current_model, current_scaler, Xr, yr,
                features_used=active_features, label=f'meta_w{week_num:03d}'
            )
            if meta_new is not None:
                current_meta, current_meta_scaler = meta_new, meta_s_new

            del rt_df, Xr, yr, yr_r, Xs_rt
            gc.collect()
            did_retrain = True
        except Exception as e:
            print(f'    Retrain failed: {e}')

    # Predict this week
    predictions, actual_rets, meta_confs = predict_week(
        current_model, current_scaler, ALL_DATA, ws, we,
        active_features, meta_model=current_meta, meta_scaler=current_meta_scaler
    )

    if len(predictions) < 5:
        weekly_returns_hist.append(0.0)
        weekly_summary_list.append({
            'week': week_num, 'week_start': str(ws.date()), 'week_end': str(we.date()),
            'n_symbols': len(predictions), 'n_trades': 0, 'week_return_pct': 0.0,
            'ic': 0.0, 'turnover_pct': 0.0, 'regime': regime, 'retrained': did_retrain,
        })
        continue

    # Trade simulation
    trades, week_ret, cur_longs, cur_shorts = simulate_weekly_trades(
        predictions, actual_rets, prev_longs, prev_shorts,
        meta_confs, weekly_returns_hist
    )
    weekly_returns_hist.append(week_ret)

    # IC
    common = [s for s in predictions if s in actual_rets]
    if len(common) > 10:
        pred_arr = np.array([predictions[s] for s in common])
        ret_arr_ = np.array([actual_rets[s] for s in common])
        week_ic  = float(sp_stats.spearmanr(pred_arr, ret_arr_)[0])
    else:
        week_ic = 0.0

    # Turnover
    n_cur = len(cur_longs) + len(cur_shorts)
    n_new = len(cur_longs - prev_longs) + len(cur_shorts - prev_shorts)
    turnover = round(n_new / max(n_cur, 1) * 100, 1)
    prev_longs, prev_shorts = cur_longs, cur_shorts

    # Cumulative stats
    cum_ret = float(np.prod([1 + r for r in weekly_returns_hist]) - 1)
    max_dd  = compute_drawdown(weekly_returns_hist)

    for t in trades:
        t['week'] = week_num
        t['week_start'] = str(ws.date())
    all_trades_list.extend(trades)

    zone = 'Y2' if we <= Y2_END else 'Y3'
    ann_proj = ((1 + week_ret) ** 52 - 1) * 100

    weekly_summary_list.append({
        'week': week_num, 'week_start': str(ws.date()), 'week_end': str(we.date()),
        'n_symbols': len(predictions), 'n_trades': len(trades),
        'week_return_pct': round(week_ret * 100, 4),
        'annualised_pct': round(ann_proj, 2), 'ic': round(week_ic, 5),
        'turnover_pct': turnover, 'cum_return_pct': round(cum_ret * 100, 4),
        'max_drawdown_pct': round(max_dd * 100, 4),
        'regime': regime, 'retrained': did_retrain,
    })

    # Progress
    if week_num % 4 == 0 or week_num <= 2:
        rolling = np.mean(weekly_returns_hist[-4:]) * 100 if weekly_returns_hist else 0
        print(f'  Week {week_num:3d} [{zone}] | ret={week_ret*100:+.2f}%  IC={week_ic:+.4f}  '
              f'cum={cum_ret*100:+.1f}%  DD={max_dd*100:.1f}%  n={len(trades)}  {regime}')

    if week_num % 13 == 0:
        print(f'  [Resource] RAM={mem_gb():.1f}GB  Elapsed={( time.time()-t0_wf)/60:.0f}min')
        gc.collect()

print(f'\n[Cell 9] v4 Walk-forward complete.')
print(f'  Weeks: {len(weekly_summary_list)} | Trades: {len(all_trades_list)} | Retrains: {retrain_count}')
if kill_switch_hit:
    print(f'  Kill-switch was triggered during run.')
gc.collect()

In [ ]:
# ── Cell 10: v4 Results + Metrics ─────────────────────────────────────────────
print('[Cell 10] Saving v4 results...')

if not weekly_summary_list:
    print('[WARN] No weekly results. Did Cell 9 run?')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)
    weekly_df.to_csv(f'{RESULTS_DIR}/weekly_summary_v4.csv', index=False)
    trades_df.to_csv(f'{RESULTS_DIR}/all_trades_v4.csv', index=False)

    rets   = weekly_df['week_return_pct'].dropna() / 100
    pnls   = trades_df['pnl_percent'].dropna() / 100 if len(trades_df) > 0 else pd.Series(dtype=float)
    ic_s   = weekly_df['ic'].dropna()
    cum    = (1 + rets).cumprod()
    n_wks  = max(len(rets), 1)
    t_ret  = float(cum.iloc[-1] - 1) if len(cum) > 0 else 0.0
    ann_ret = (1 + t_ret) ** (52 / n_wks) - 1
    sharpe  = float(rets.mean() / (rets.std() + 1e-9) * np.sqrt(52))
    max_dd  = float(((cum - cum.cummax()) / (1 + cum.cummax())).min()) if len(cum) > 0 else 0.0
    win_rt  = float((pnls > 0).mean() * 100) if len(pnls) > 0 else 0.0
    ic_mean = float(ic_s.mean()) if len(ic_s) > 0 else 0.0
    ic_std  = float(ic_s.std())  if len(ic_s) > 0 else 0.0
    icir_v  = float(ic_mean / (ic_std + 1e-8))
    ic_pos  = float((ic_s > 0).mean() * 100) if len(ic_s) > 0 else 0.0
    avg_to  = float(weekly_df['turnover_pct'].mean()) if 'turnover_pct' in weekly_df.columns else 0.0

    # VaR / CVaR (inline — no RiskManager import on Kaggle)
    ret_series = pd.Series([r/100 for r in weekly_df['week_return_pct'].dropna()])
    if len(ret_series) > 4:
        var_95  = float(ret_series.quantile(1 - VAR_CONFIDENCE))
        cvar_95 = float(ret_series[ret_series <= var_95].mean()) if (ret_series <= var_95).any() else var_95
    else:
        var_95 = cvar_95 = 0.0

    # Y2 vs Y3 split analysis
    y2_mask = weekly_df['week_end'].apply(
        lambda x: pd.Timestamp(x, tz='UTC') <= Y2_END if x else False)
    y3_mask = ~y2_mask
    y2_ret = float(weekly_df.loc[y2_mask, 'week_return_pct'].sum()) if y2_mask.any() else 0.0
    y3_ret = float(weekly_df.loc[y3_mask, 'week_return_pct'].sum()) if y3_mask.any() else 0.0
    y2_ic  = float(weekly_df.loc[y2_mask, 'ic'].mean()) if y2_mask.any() else 0.0
    y3_ic  = float(weekly_df.loc[y3_mask, 'ic'].mean()) if y3_mask.any() else 0.0

    performance = {
        'label': 'v4_WalkForward_Y2Y3',
        'total_weeks': n_wks, 'total_trades': len(trades_df), 'retrains': retrain_count,
        'total_return_pct': round(t_ret * 100, 2),
        'annualised_pct': round(ann_ret * 100, 2),
        'sharpe': round(sharpe, 4),
        'max_drawdown_pct': round(max_dd * 100, 2),
        'win_rate_pct': round(win_rt, 2),
        'ic_mean': round(ic_mean, 5),
        'icir': round(icir_v, 4),
        'ic_positive_pct': round(ic_pos, 1),
        'avg_turnover_pct': round(avg_to, 1),
        'var_95': round(var_95 * 100, 4),
        'cvar_95': round(cvar_95 * 100, 4),
        'y2_return_pct': round(y2_ret, 2),
        'y3_return_pct': round(y3_ret, 2),
        'y2_ic_mean': round(y2_ic, 5),
        'y3_ic_mean': round(y3_ic, 5),
        'meta_labeling': META_MODEL is not None,
        'kill_switch': MAX_DRAWDOWN_KILL,
        'use_gpu': HAS_GPU,
    }
    with open(f'{RESULTS_DIR}/performance_v4.json', 'w') as f:
        json.dump(performance, f, indent=2)

    print(f'\n  ── v4 Walk-Forward Performance (Y2+Y3 OOS) ──')
    print(f'  Total Return   : {performance["total_return_pct"]:>8.2f}%')
    print(f'  Annualised     : {performance["annualised_pct"]:>8.2f}%')
    print(f'  Sharpe         : {performance["sharpe"]:>8.4f}')
    print(f'  Max Drawdown   : {performance["max_drawdown_pct"]:>8.2f}%')
    print(f'  Win Rate       : {performance["win_rate_pct"]:>8.2f}%')
    print(f'  IC Mean        : {performance["ic_mean"]:>8.5f}')
    print(f'  ICIR           : {performance["icir"]:>8.4f}')
    print(f'  VaR (95%)      : {performance["var_95"]:>8.4f}%')
    print(f'  CVaR (95%)     : {performance["cvar_95"]:>8.4f}%')
    print(f'  Y2 Return      : {performance["y2_return_pct"]:>8.2f}%  (IC={y2_ic:.4f})')
    print(f'  Y3 Return      : {performance["y3_return_pct"]:>8.2f}%  (IC={y3_ic:.4f})')
    print(f'  Avg Turnover   : {performance["avg_turnover_pct"]:>8.1f}%')
    print(f'  Retrains       : {retrain_count}')
    print(f'  Meta-labeling  : {performance["meta_labeling"]}')

In [ ]:
# ── Cell 11: v4 Charts + ZIP ──────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import zipfile

print('[Cell 11] Generating v4 performance charts...')

if not weekly_summary_list:
    print('[WARN] No results to chart.')
else:
    weekly_df = pd.DataFrame(weekly_summary_list)
    trades_df = pd.DataFrame(all_trades_list)

    fig = plt.figure(figsize=(16, 11))
    fig.suptitle('Azalyst v4.0 — Walk-Forward Y2+Y3 (Expanding Window + IC Selection + Kill-Switch)',
                 fontsize=13, fontweight='bold')
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

    # Panel 1: Cumulative return with Y2/Y3 zones
    ax1 = fig.add_subplot(gs[0, 0])
    rets = weekly_df['week_return_pct'].fillna(0) / 100
    cum  = ((1 + rets).cumprod() - 1) * 100
    ax1.plot(weekly_df['week'], cum, color='#1f77b4', linewidth=2)
    ax1.fill_between(weekly_df['week'], cum, alpha=0.12, color='#1f77b4')
    # Mark Y2/Y3 boundary
    y2_weeks = len(weekly_df[weekly_df['week_end'].apply(
        lambda x: pd.Timestamp(x, tz='UTC') <= Y2_END if x else False)])
    if 0 < y2_weeks < len(weekly_df):
        ax1.axvline(y2_weeks, color='red', linewidth=1.5, linestyle='--', alpha=0.6, label='Y2→Y3')
        ax1.legend(fontsize=9)
    ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax1.set_title('Cumulative Return (%)', fontweight='bold')
    ax1.set_xlabel('Week #'); ax1.set_ylabel('%'); ax1.grid(True, alpha=0.25)

    # Panel 2: Weekly return distribution
    ax2 = fig.add_subplot(gs[0, 1])
    wr = weekly_df['week_return_pct'].dropna()
    ax2.hist(wr, bins=min(30, max(10, len(wr)//3)), color='#ff7f0e', alpha=0.72,
             edgecolor='black', linewidth=0.4)
    if len(wr) > 2:
        ax2.axvline(wr.mean(),   color='red',   linewidth=1.8, linestyle='--', label=f'Mean {wr.mean():.2f}%')
        ax2.axvline(wr.median(), color='green', linewidth=1.2, linestyle=':',  label=f'Median {wr.median():.2f}%')
        ax2.legend(fontsize=9)
    ax2.set_title('Weekly Return Distribution', fontweight='bold')
    ax2.set_xlabel('Weekly Return (%)'); ax2.set_ylabel('Count'); ax2.grid(True, alpha=0.25)

    # Panel 3: IC series
    ax3 = fig.add_subplot(gs[1, 0])
    ic_vals = weekly_df['ic'].fillna(0)
    ax3.bar(weekly_df['week'], ic_vals,
            color=['#2ca02c' if v > 0 else '#d62728' for v in ic_vals], alpha=0.75, width=0.8)
    if len(ic_vals) > 2:
        ax3.axhline(ic_vals.mean(), color='navy', linewidth=1.5, linestyle='--',
                    label=f'Mean IC {ic_vals.mean():.4f}')
        ax3.legend(fontsize=9)
    ax3.axhline(0, color='black', linewidth=0.6)
    ax3.set_title('Weekly IC (Information Coefficient)', fontweight='bold')
    ax3.set_xlabel('Week #'); ax3.set_ylabel('Spearman IC'); ax3.grid(True, alpha=0.25)

    # Panel 4: Trade P&L
    ax4 = fig.add_subplot(gs[1, 1])
    if len(trades_df) > 0 and 'pnl_percent' in trades_df.columns:
        pnl = trades_df['pnl_percent'].dropna()
        ax4.hist(pnl, bins=min(40, max(10, len(pnl)//20)), color='#9467bd', alpha=0.72,
                 edgecolor='black', linewidth=0.3)
        ax4.axvline(pnl.mean(), color='red', linewidth=1.8, linestyle='--',
                    label=f'Mean {pnl.mean():.3f}%')
        ax4.axvline(0, color='black', linewidth=0.8)
        ax4.legend(fontsize=9)
        ax4.set_title(f'Trade P&L Distribution (n={len(pnl):,})', fontweight='bold')
        ax4.set_xlabel('P&L per Trade (%)'); ax4.set_ylabel('Count'); ax4.grid(True, alpha=0.25)

    chart_path = f'{RESULTS_DIR}/performance_v4.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  Chart saved -> {chart_path}')

    # ZIP results
    zip_path = f'{RESULTS_DIR}/azalyst_v4_results.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fp in Path(RESULTS_DIR).rglob('*'):
            if fp.is_file() and fp.suffix in ('.csv', '.json', '.png', '.pkl'):
                zf.write(fp, fp.relative_to(Path(RESULTS_DIR).parent))
    zip_mb = Path(zip_path).stat().st_size / 1e6
    print(f'  Results ZIP -> {zip_path}  ({zip_mb:.1f} MB)')

    print(f'''
=========================================================
  AZALYST v4.0 — RUN COMPLETE
  Built by Azalyst Research
=========================================================
  Total Return   : {performance["total_return_pct"]:.2f}%
  Annualised     : {performance["annualised_pct"]:.2f}%
  Sharpe         : {performance["sharpe"]:.4f}
  Max Drawdown   : {performance["max_drawdown_pct"]:.2f}%
  IC Mean        : {performance["ic_mean"]:.5f}
  ICIR           : {performance["icir"]:.4f}
  VaR (95%)      : {performance["var_95"]:.4f}%
  CVaR (95%)     : {performance["cvar_95"]:.4f}%
  Y2 Return      : {performance["y2_return_pct"]:.2f}%
  Y3 Return      : {performance["y3_return_pct"]:.2f}%
  Kill-Switch    : {MAX_DRAWDOWN_KILL*100:.0f}%
  Meta-labeling  : {performance["meta_labeling"]}
  GPU used       : {HAS_GPU}
=========================================================
  v4 Architecture:
    1. Expanding Window Training
    2. Regime-Aware IC Feature Selection
    3. Risk Integration (VaR/CVaR)
    4. Drawdown Kill-Switch
    5. SHAP Explainability
=========================================================
  Download azalyst_v4_results.zip from Output tab.
''')